# 03 · One-stage anchor-based detector: RetinaNet

## Setup

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-object-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwdet import env
from hwdet.data.download import ensure_data
from hwdet.tracker import Run, status
P = env.setup(seed=42)
ensure_data(P, labels_only=False)   # True on the Mac: annotations only
status(P)                           # which runs / stages are already done

In [ ]:
# Imports, plot style, W&B login (the key is read from Kaggle / Colab secrets; without it W&B is skipped)
import math
import types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn

from hwdet import CLASS_NAMES, NUM_CLASSES
from hwdet.config import load_config
from hwdet.data.detection_dataset import DetectionDataset, build_targets
from hwdet.data.io import frame_table
from hwdet.engine import Data, run_experiment
from hwdet.env import REPO_ROOT
from hwdet.eval.coco_eval import evaluate
from hwdet.tracker import load_registry
from hwdet.viz import INK_3, PALETTE, save_fig, set_style
from hwdet.wandb_utils import login_from_secrets

set_style()
login_from_secrets()
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", env.device())

## 1. Data
Same protocol as Faster R-CNN (`configs/experiments/_protocol.yaml`): every 10th training frame, the full
official val split for final metrics, every 5th val frame after each epoch, 8 epochs at batch size 4.

In [ ]:
# Frames and targets (COCO annotations -> torchvision boxes/labels + ignored regions), built once
protocol = load_config("retina_baseline")["split"]
train_frames = frame_table(P.data, "train")
train_frames = train_frames[(train_frames.frame - 1) % protocol["train_frame_step"] == 0]
val_frames = frame_table(P.data, "val")
monitor_frames = val_frames[(val_frames.frame - 1) % protocol["monitor_frame_step"] == 0]
test_frames = frame_table(P.data, "test")
targets = build_targets(P.data, pd.concat([train_frames, val_frames]))
data = Data(P.data, train_frames, val_frames, monitor_frames, targets, test_frames)
pd.DataFrame({"frames": [len(train_frames), len(val_frames), len(monitor_frames), len(test_frames)],
              "clips": [d["clip"].nunique() for d in (train_frames, val_frames, monitor_frames, test_frames)]},
             index=["train (subsampled)", "val", "val monitor", "test"])

## 2. Model
torchvision RetinaNet v2 (ResNet-50 + FPN, GroupNorm heads, GIoU box loss), pretrained on COCO. There is
no proposal stage: a classification head and a box head run on every anchor of every FPN level (P3–P7,
strides 8–128). Classification uses sigmoid outputs trained with the focal loss.

The COCO classification layer is replaced with a new one for 9 outputs (index 0 unused, so labels stay
1–8 as for Faster R-CNN), initialised with the prior probability 0.01 as in the RetinaNet paper. The
pretrained convolution towers are kept. torchvision fixes the focal loss at α = 0.25, γ = 2, so the
classification loss is replaced by the same function with α and γ read from the config.

In [ ]:
# Builds RetinaNet from a config: COCO weights, new 9-output classifier, anchors, matcher IoUs, focal loss
from torchvision.models.detection import RetinaNet_ResNet50_FPN_V2_Weights, retinanet_resnet50_fpn_v2
from torchvision.models.detection._utils import Matcher
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import sigmoid_focal_loss

MODEL_KWARGS = ["min_size", "max_size", "score_thresh", "nms_thresh", "detections_per_img",
                "fg_iou_thresh", "bg_iou_thresh", "topk_candidates"]


def anchor_generator(m):
    sizes = tuple(tuple(s * k for k in m["anchor_scales"]) for s in m["anchor_sizes"])
    return AnchorGenerator(sizes, (tuple(m["aspect_ratios"]),) * len(sizes))


def focal_classification_loss(self, targets, head_outputs, matched_idxs):
    """torchvision's RetinaNet classification loss, with alpha / gamma taken from the head."""
    losses = []
    for tgt, logits, matched in zip(targets, head_outputs["cls_logits"], matched_idxs):
        fg = matched >= 0
        gt = torch.zeros_like(logits)
        gt[fg, tgt["labels"][matched[fg]]] = 1.0
        valid = matched != Matcher.BETWEEN_THRESHOLDS                 # anchors between the IoU thresholds are ignored
        loss = sigmoid_focal_loss(logits[valid], gt[valid], alpha=self.focal_alpha, gamma=self.focal_gamma,
                                  reduction="sum")
        losses.append(loss / max(1, fg.sum()))
    return sum(losses) / len(targets)


def new_conv(out_channels, bias):
    conv = nn.Conv2d(256, out_channels, kernel_size=3, padding=1)
    nn.init.normal_(conv.weight, std=0.01)
    nn.init.constant_(conv.bias, bias)
    return conv


def build_retina(cfg):
    m = cfg["model"]
    model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1,
                                      **{k: m[k] for k in MODEL_KWARGS if k in m})
    model.anchor_generator = anchor_generator(m)
    n_anchors = model.anchor_generator.num_anchors_per_location()[0]
    cls_head, reg_head = model.head.classification_head, model.head.regression_head
    cls_head.cls_logits = new_conv(n_anchors * (NUM_CLASSES + 1), -math.log((1 - 0.01) / 0.01))
    cls_head.num_classes, cls_head.num_anchors = NUM_CLASSES + 1, n_anchors
    if reg_head.bbox_reg.out_channels != n_anchors * 4:              # new anchor count -> new box layer
        reg_head.bbox_reg = new_conv(n_anchors * 4, 0.0)
    cls_head.focal_alpha, cls_head.focal_gamma = m["focal_alpha"], m["focal_gamma"]
    cls_head.compute_loss = types.MethodType(focal_classification_loss, cls_head)
    return model

## 3. Anchors and object sizes
Objects are measured after the model's resize (shorter side 800, longer side ≤ 1333), on 150 training
frames. An object is *covered* when some anchor overlaps it with IoU ≥ the positive threshold. Objects
below it still get their single best anchor as a positive (torchvision's low-quality matches), but only
one. The same pass counts how many of all anchors are positive, negative or ignored per image.

In [ ]:
# Anchor coverage and positive / negative anchor counts for the default and the modified anchor settings
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.ops import box_iou

SIZE_BINS, SIZE_LABELS = [0, 16, 32, 96, 1e5], ["< 16 px", "16–32 px", "32–96 px", "> 96 px"]


def feature_sizes(h, w):
    sizes = [(h // s, w // s) for s in (8, 16, 32)]                   # padded input is divisible by 32
    for _ in range(2):                                                # P6, P7: stride-2 convs on top of P5
        sizes.append((math.ceil(sizes[-1][0] / 2), math.ceil(sizes[-1][1] / 2)))
    return sizes


def anchor_stats(config, frames, n=150):
    m = load_config(config)["model"]
    tf = GeneralizedRCNNTransform(m["min_size"], m["max_size"], [0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225]).eval()
    gen, dev = anchor_generator(m), env.device()
    ds = DetectionDataset(frames, P.data, targets)
    objs, imgs = [], []
    for i in np.linspace(0, len(ds) - 1, n).astype(int):
        img, tgt, _ = ds[i]
        if len(tgt["boxes"]) == 0:
            continue
        il, t = tf([img], [{"boxes": tgt["boxes"], "labels": tgt["labels"]}])
        feats = [torch.empty(1, 1, h, w) for h, w in feature_sizes(*il.tensors.shape[-2:])]
        anchors = gen(il, feats)[0].to(dev)
        boxes = t[0]["boxes"].to(dev)
        iou = box_iou(boxes, anchors)                                 # objects x anchors
        wh = boxes[:, 2:] - boxes[:, :2]
        objs.append(pd.DataFrame({"side": wh.prod(1).sqrt().cpu().numpy(),
                                  "covered": (iou.max(1).values >= m["fg_iou_thresh"]).cpu().numpy(),
                                  "positives": (iou >= m["fg_iou_thresh"]).sum(1).cpu().numpy()}))
        best = iou.max(0).values
        imgs.append({"anchors": len(anchors), "positive": int((best >= m["fg_iou_thresh"]).sum()),
                     "ignored": int(((best >= m["bg_iou_thresh"]) & (best < m["fg_iou_thresh"])).sum())})
    objs = pd.concat(objs, ignore_index=True)
    objs["size"] = pd.cut(objs.side, SIZE_BINS, labels=SIZE_LABELS)
    return objs, pd.DataFrame(imgs)


stats = {c: anchor_stats(c, train_frames) for c in ["retina_baseline", "retina_anchors", "retina_iou"]}
objs0 = stats["retina_baseline"][0]
coverage = pd.DataFrame({c: o.groupby("size", observed=False).covered.mean() for c, (o, _) in stats.items()}).T
coverage.insert(0, "all objects", [o.covered.mean() for o, _ in stats.values()])
coverage.loc["share of objects"] = [1.0] + list(objs0["size"].value_counts(normalize=True).reindex(SIZE_LABELS))
display((coverage * 100).round(1))
pd.DataFrame({c: {"anchors per image": im.anchors.mean(),
                  "positive anchors per image": im.positive.mean(),
                  "positive share, %": 100 * (im.positive / im.anchors).mean(),
                  "ignored share, %": 100 * (im.ignored / im.anchors).mean(),
                  "positives per object (median)": o.positives.median()}
              for c, (o, im) in stats.items()}).round(3)

In [ ]:
# Object sizes after resizing vs the smallest and largest anchor of each anchor setting
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.hist(objs0.side, bins=np.logspace(np.log10(4), np.log10(600), 60), color=PALETTE[0], alpha=0.85)
ax.set_xscale("log")
for k, (col, name) in enumerate([(PALETTE[1], "retina_baseline"), (PALETTE[2], "retina_anchors")]):
    m = load_config(name)["model"]
    lo, hi = m["anchor_sizes"][0], m["anchor_sizes"][-1] * m["anchor_scales"][-1]
    for x in (lo, hi):
        ax.axvline(x, color=col, lw=2, ls="--")
    ax.text(lo * 1.04, ax.get_ylim()[1] * (0.92 - 0.1 * k), f"{name}: {lo:.0f}–{hi:.0f} px", color=col, fontsize=9)
ax.set(xlabel="object side √(w·h) after resizing, px (log scale)", ylabel="objects",
       title="Object sizes seen by the model and the anchor range")
fig.tight_layout()
save_fig(fig, "retina_anchor_sizes", P)

## 4. Smoke test
One short epoch on a few frames under a separate run id, to catch errors before the long runs. It is
deleted afterwards.

In [ ]:
# Smoke test: 1 epoch on ~40 frames, then the run folder and its registry row are removed
import shutil
smoke = Data(P.data, train_frames.iloc[::60], val_frames.iloc[::150], val_frames.iloc[::150], targets)
run_experiment("retina_baseline", build_retina, P, smoke, run_id="smoke_retina", wandb=False,
               train={"epochs": 1, "warmup_iters": 5})
shutil.rmtree(P.runs / "smoke_retina", ignore_errors=True)
reg_file = P.outputs / "experiments.csv"
if reg_file.exists():
    reg = pd.read_csv(reg_file)
    reg[reg.run_id != "smoke_retina"].to_csv(reg_file, index=False)

## 5. Experiments
Each cell trains one config, evaluates the best epoch on the full val split and saves everything under
`outputs/runs/<config>/`. Re-running a cell after a crash continues where it stopped.

| config | change vs baseline |
|---|---|
| `retina_baseline` | torchvision defaults, lr 0.01, 300 detections per image |
| `retina_anchors` | anchors 16–406 px instead of 32–813 px (P3 base 16) |
| `retina_iou` | positive IoU ≥ 0.4, negative < 0.3 (default 0.5 / 0.4) |
| `retina_focal` | focal loss γ = 1.5 instead of 2 |

In [ ]:
# Baseline
runs = {}
runs["retina_baseline"] = run_experiment("retina_baseline", build_retina, P, data)

In [ ]:
# Experiment 1: anchors from the object sizes
runs["retina_anchors"] = run_experiment("retina_anchors", build_retina, P, data)

In [ ]:
# Experiment 2: IoU thresholds for positive / negative anchors
runs["retina_iou"] = run_experiment("retina_iou", build_retina, P, data)

In [ ]:
# Experiment 3: focal loss gamma
runs["retina_focal"] = run_experiment("retina_focal", build_retina, P, data)

## 6. Tuned model
The tuned model combines every change that raised val mAP by at least 0.005 on its own (smaller
differences are within run-to-run noise, see the Faster R-CNN runs). With fewer than two such changes
nothing new is trained: the tuned model is the best single run.

In [ ]:
# Tuned RetinaNet: baseline + all changes that helped; test predictions + submission for it
EXPERIMENTS = ["retina_anchors", "retina_iou", "retina_focal"]
val = load_registry(P).query("eval == 'val'").set_index("run_id")
gain = {r: val.loc[r, "mAP"] - val.loc["retina_baseline", "mAP"] for r in EXPERIMENTS}
helped = [r for r in EXPERIMENTS if gain[r] >= 0.005]
print("mAP change vs baseline:", {r: round(g, 4) for r, g in gain.items()})

if len(helped) >= 2:
    base_model = load_config("retina_baseline")["model"]
    changes = {k: v for r in helped for k, v in load_config(r)["model"].items() if base_model[k] != v}
    tuned_id = "retina_tuned"
    run_experiment("retina_baseline", build_retina, P, data, predict_test=True, run_id=tuned_id,
                   model=changes, description="RetinaNet tuned: " + " + ".join(helped))
else:
    tuned_id = max(["retina_baseline"] + helped, key=lambda r: val.loc[r, "mAP"])
    run_experiment(tuned_id, build_retina, P, data, predict_test=True)
print("tuned model:", tuned_id)

## 7. Comparison with the tuned two-stage detector
The two-stage reference is the Faster R-CNN run with the best val mAP (`results/experiments.csv`, from
notebook 02). All numbers are on the full val split.

In [ ]:
# Val metrics: tuned Faster R-CNN, all RetinaNet runs (committed results + runs of this notebook)
reg = (pd.concat([pd.read_csv(REPO_ROOT / "results/experiments.csv"), load_registry(P)])
       .sort_values("updated").drop_duplicates(["run_id", "eval"], keep="last"))
val = reg[reg["eval"] == "val"].set_index("run_id")
frcnn_id = val[val.family == "two_stage"].mAP.idxmax()
order = [frcnn_id, "retina_baseline"] + EXPERIMENTS + (["retina_tuned"] if tuned_id == "retina_tuned" else [])
cols = ["description", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms_per_img"]
val.loc[order, cols].round(4)

In [ ]:
# Per-class AP (%)
(val.loc[order, [f"AP_{c}" for c in CLASS_NAMES]].rename(columns=lambda c: c[3:]) * 100).round(1)

In [ ]:
# AP by object size: tuned Faster R-CNN vs initial and tuned RetinaNet
models = list(dict.fromkeys([frcnn_id, "retina_baseline", tuned_id]))
metrics = ["AP_small", "AP_medium", "AP_large", "mAP"]
fig, ax = plt.subplots(figsize=(10, 3.8))
width = 0.8 / len(models)
for k, (r, col) in enumerate(zip(models, PALETTE)):
    xs = np.arange(len(metrics)) + (k - (len(models) - 1) / 2) * width
    ax.bar(xs, val.loc[r, metrics].astype(float), width * 0.92, color=col, label=r)
    for x, v in zip(xs, val.loc[r, metrics].astype(float)):
        ax.text(x, v + 0.005, f"{v:.3f}", ha="center", fontsize=8, color=INK_3)
ax.set_xticks(np.arange(len(metrics)), ["small", "medium", "large", "all (mAP)"])
ax.set(ylabel="AP (IoU 0.50:0.95)", title="Val AP by object size")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
save_fig(fig, "retina_vs_frcnn_sizes", P)

In [ ]:
# Training curves: val mAP (monitor subset) per epoch, RetinaNet runs vs the tuned Faster R-CNN
def history(run_id):
    f = P.runs / run_id / "history.csv"
    return pd.read_csv(f if f.exists() else REPO_ROOT / "results/runs" / run_id / "history.csv")


fig, (a0, a1) = plt.subplots(1, 2, figsize=(14, 4.2))
for r, col in zip(order, PALETTE):
    h = history(r)
    loss_cols = [c for c in h.columns if c.startswith("loss_") or c in ("classification", "bbox_regression")]
    ls = "--" if r == frcnn_id else "-"
    a0.plot(h.epoch, h[loss_cols].sum(1), marker="o", color=col, ls=ls, label=r)
    a1.plot(h.epoch, h.val_mAP, marker="o", color=col, ls=ls, label=r)
a0.set(title="Training loss (sum of the model's losses)", xlabel="epoch")
a1.set(title="Val mAP (monitor subset)", xlabel="epoch")
a1.legend(loc="lower right")
fig.tight_layout()
save_fig(fig, "retina_curves", P)

**Dense scenes.** Val frames are split by the number of annotated objects, and AP is computed per group
from the saved val predictions. The Faster R-CNN predictions are available when the output of
notebook 02 is attached as an input (restored by the setup cell).

In [ ]:
# AP by scene density (objects per frame) from saved val predictions
n_obj = val_frames.file_name.map(lambda f: len(targets[f]["boxes"]))
density = pd.cut(n_obj, [-1, 20, 60, 1000], labels=["≤ 20 objects", "21–60", "> 60"])
rows = {}
for r in models:
    f = P.runs / r / "predictions" / "val.csv"
    if not f.exists():
        print(f"{r}: no val predictions in {P.runs}")
        continue
    preds = pd.read_csv(f)
    for d in density.cat.categories:
        fr = val_frames[density == d]
        m = evaluate(preds[preds.file_name.isin(fr.file_name)], fr, P.data, per_class=False)
        rows[(r, d)] = {"frames": len(fr), "mAP": m["mAP"], "AP50": m["AP50"], "AP_small": m["AP_small"]}
pd.DataFrame(rows).T.round(4)